# Evaluating LLMs: Metrics, Benchmarks, Judges and Error Bars

The full code of [Evaluating LLMs: Metrics, Benchmarks, Judges and Error Bars](https://residual-stream.lvtan71.workers.dev/posts/evaluating-llms/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/evaluating-llms) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/evaluating-llms.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 3 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 3 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "evaluating-llms"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/evaluating-llms


## `sample_size.py`

How many questions it takes to detect 70% against 72%, unpaired and paired; Wilson intervals for one accuracy; the interval on a difference at 500 questions; Cohen's kappa for two raters.

In [2]:
!{sys.executable} sample_size.py

unpaired (different questions): 8,077 questions per model
paired, models disagree on 30% of questions: 5,887 questions
paired, models disagree on 15% of questions: 2,944 questions
paired, models disagree on 8% of questions: 1,570 questions

70 right of 100: accuracy 70%, 95% interval 60.4% to 78.1% (about ±8.8 points)
700 right of 1,000: accuracy 70%, 95% interval 67.1% to 72.8% (about ±2.8 points)
7,000 right of 10,000: accuracy 70%, 95% interval 69.1% to 70.9% (about ±0.9 points)

500 questions, 95% interval on the difference in accuracy: unpaired ±5.6 points, paired (disagreeing on 15%) ±3.4 points

Cohen's kappa: observed agreement 0.85, chance agreement 0.55 x 0.50 + 0.45 x 0.50 = 0.500, kappa = (0.85 - 0.500) / (1 - 0.500) = 0.70


The post printed:

```text
unpaired (different questions): 8,077 questions per model
paired, models disagree on 30% of questions: 5,887 questions
paired, models disagree on 15% of questions: 2,944 questions
paired, models disagree on 8% of questions: 1,570 questions

70 right of 100: accuracy 70%, 95% interval 60.4% to 78.1% (about ±8.8 points)
700 right of 1,000: accuracy 70%, 95% interval 67.1% to 72.8% (about ±2.8 points)
7,000 right of 10,000: accuracy 70%, 95% interval 69.1% to 70.9% (about ±0.9 points)

500 questions, 95% interval on the difference in accuracy: unpaired ±5.6 points, paired (disagreeing on 15%) ±3.4 points

Cohen's kappa: observed agreement 0.85, chance agreement 0.55 x 0.50 + 0.45 x 0.50 = 0.500, kappa = (0.85 - 0.500) / (1 - 0.500) = 0.70
```

## `judge.py`

Qwen2.5-1.5B-Instruct as a pairwise judge on 100 multiplications and 20 capitals with known answers: position bias, and how claims of checking or neutral padding change its choice.

In [3]:
!{sys.executable} judge.py

120 questions with a known answer: 100 two-digit multiplications, 20 capitals

                                          multiplication  capitals


right answer first: judge right                      99%       85%


right answer second: judge right                     12%      100%


  same, first answer labeled B                        0%       55%


right in both orders                                 12%       85%


chose the same slot both times                       87%       15%

share of judgments picking the right answer, both orders averaged:


bare right vs bare wrong                             56%       92%


bare right vs wrong + claims of checking              6%       42%


bare right vs wrong + neutral padding                48%       82%


right + neutral padding vs bare wrong                62%       95%


The post printed:

```text
120 questions with a known answer: 100 two-digit multiplications, 20 capitals

                                          multiplication  capitals
right answer first: judge right                      99%       85%
right answer second: judge right                     12%      100%
  same, first answer labeled B                        0%       55%
right in both orders                                 12%       85%
chose the same slot both times                       87%       15%

share of judgments picking the right answer, both orders averaged:
bare right vs bare wrong                             56%       92%
bare right vs wrong + claims of checking              6%       42%
bare right vs wrong + neutral padding                48%       82%
right + neutral padding vs bare wrong                62%       95%
```

## `arena.py`

Bradley–Terry ratings fitted to simulated arena votes for four models with known ratings, at 500, 5,000 and 50,000 votes, with 95% bootstrap intervals.

In [4]:
!{sys.executable} arena.py

500 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    44, 95% interval    -9 to    93
  model C: true    35, estimated    51, 95% interval    -2 to   104
  model D: true   104, estimated   133, 95% interval    73 to   184


5,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    19, 95% interval     5 to    38
  model C: true    35, estimated    33, 95% interval    16 to    53
  model D: true   104, estimated   102, 95% interval    87 to   120


50,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    17, 95% interval    11 to    22
  model C: true    35, estimated    30, 95% interval    25 to    36
  model D: true   104, estimated   100, 95% interval    95 to   105


The post printed:

```text
500 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    44, 95% interval    -9 to    93
  model C: true    35, estimated    51, 95% interval    -2 to   104
  model D: true   104, estimated   133, 95% interval    73 to   184
5,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    19, 95% interval     5 to    38
  model C: true    35, estimated    33, 95% interval    16 to    53
  model D: true   104, estimated   102, 95% interval    87 to   120
50,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    17, 95% interval    11 to    22
  model C: true    35, estimated    30, 95% interval    25 to    36
  model D: true   104, estimated   100, 95% interval    95 to   105
```